# ANCOVA (Analysis of Covariance) Nedir?
ANCOVA, ANOVA ile Regresyonun (Konu 113+'ta detaylı göreceğiz) birleşimi — ANOVA'yı, **sürekli bir "karıştırıcı" (confounding) değişkeni kontrol ederek** çalıştırmamızı sağlar.

## Neden Gerekli?
Diyelim kanallar arası Memnuniyet farkına bakıyorsun, ama biliyorsun ki **Müşteri Yaşı** da memnuniyeti etkiliyor, VE farklı kanallarda yaş dağılımı farklı olabilir (örn: Sosyal Medya kanalına daha genç müşteriler geliyor olabilir). Bu durumda, kanallar arası fark bulduğumuzda, bu gerçekten **kanaldan mı** geliyor, yoksa aslında **yaş farkından mı** kaynaklanıyor, karışabilir.

ANCOVA, yaş gibi bir **kovaryatı (covariate)** modele ekleyerek, "yaşın etkisini istatistiksel olarak sabitleyip/kontrol ederek, kanalın SAF etkisini" ölçmemizi sağlar.

## Hipotezler
- **H0:** Kovaryat kontrol edildikten sonra, grupların düzeltilmiş ortalamaları arasında fark yoktur.
- **H1:** Kovaryat kontrol edildikten sonra, grupların düzeltilmiş ortalamaları arasında fark vardır.

## Python'da Kullanımı
Two-Way ANOVA ile birebir aynı yapı, sadece formüle sürekli bir değişken 
ekliyoruz:
```python
model = ols('Memnuniyet ~ C(Kanal) + Yas', data=df).fit()
print(sm.stats.anova_lm(model, typ=2))
```
Dikkat: `Yas` etrafında `C()` YOK — çünkü bu sürekli/sayısal bir değişken, 
kategorik değil.

## ANCOVA'nın Ek Varsayımı: Regresyon Eğimlerinin Homojenliği
Kovaryatın (Yaş), her grupta **benzer şekilde** etkili olması gerekir yani "yaş arttıkça memnuniyet artıyor" ilişkisinin eğimi, tüm kanallarda kabaca aynı olmalı. Bu, `C(Kanal):Yas` etkileşim terimini modele ekleyip anlamlı çıkıp çıkmadığına bakarak kontrol edilir (anlamlıysa varsayım ihlal edilmiş demektir).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(42)
n = 30

kanal_bilgisi = {
    'Telefon':      {'yas_ort': 50, 'memnuniyet_taban': 70},
    'Canlı Sohbet': {'yas_ort': 25, 'memnuniyet_taban': 70},
    'Email':        {'yas_ort': 35, 'memnuniyet_taban': 70},
}

kanallar, yaslar, memnuniyetler = [], [], []
for kanal, bilgi in kanal_bilgisi.items():
    yas = np.random.normal(bilgi['yas_ort'], 8, n)
    # Memnuniyet, aslında YAŞA bağlı (kanaldan bağımsız gerçek bir ilişki)
    memnuniyet = bilgi['memnuniyet_taban'] + (yas - 35) * 0.5 + np.random.normal(0, 6, n)
    kanallar += [kanal]*n
    yaslar += list(yas)
    memnuniyetler += list(memnuniyet)

df = pd.DataFrame({'Kanal': kanallar, 'Yas': yaslar, 'Memnuniyet': memnuniyetler})

model_anova = ols('Memnuniyet ~ C(Kanal)', data=df).fit()
print("Normal ANOVA (Yaş kontrol edilmeden):")
print(sm.stats.anova_lm(model_anova, typ=2))

model_ancova = ols('Memnuniyet ~ C(Kanal) + Yas', data=df).fit()
print("\nANCOVA (Yaş kontrol edilerek):")
print(sm.stats.anova_lm(model_ancova, typ=2))

Normal ANOVA (Yaş kontrol edilmeden):
               sum_sq    df          F        PR(>F)
C(Kanal)  1858.328487   2.0  22.312234  1.506413e-08
Residual  3623.002967  87.0        NaN           NaN

ANCOVA (Yaş kontrol edilerek):
               sum_sq    df          F        PR(>F)
C(Kanal)    85.741411   2.0   1.378795  2.573931e-01
Yas        949.015013   1.0  30.521937  3.461555e-07
Residual  2673.987954  86.0        NaN           NaN


### Sonuç
İlk başta yaptığımız tek yönlü varyans analizi (ANOVA) sonuçlarına göre, 
kanallardan en az birinin ortalama memnuniyeti diğerlerinden anlamlı 
biçimde farklılaşmaktaydı (p<0.001). Ancak yaş değişkeninin etkisini 
sabitleyip (ANCOVA ile) sadece kanallar arası memnuniyet farkını 
incelediğimizde, kanallar arasında artık anlamlı bir fark bulunamadı 
(p=0.257), buna karşılık Yaş değişkeni tek başına anlamlı bir etkiye 
sahip çıktı (p<0.001).

Bu durum, ilk ANOVA'da gözlemlenen "kanal farkının" aslında gerçek bir 
kanal etkisi olmadığını, kanalların doğal olarak farklı yaş gruplarına 
hitap etmesinden (Telefon'da yaşlı, Canlı Sohbet'te genç müşteriler 
ağırlıkta) kaynaklanan bir "karıştırıcı değişken (confounding variable)" 
etkisi olduğunu göstermektedir. Yani memnuniyeti asıl belirleyen kanalın 
kendisi değil, o kanalı kullanan müşterilerin yaşıdır. Bu bulgu, iş 
kararlarında yalnızca gözlemlenen farklara değil, bu farkların altında 
yatan gerçek nedenlere de bakmanın önemini göstermektedir.